<a href="https://colab.research.google.com/github/ALiao18/SUDC/blob/main/ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Interpretable Clinical Modeling — Prone Positioning (SUDC)

**Target:** `child_face_position_fi___1` (1 = prone face down)  
**CV Protocol:** StratifiedKFold k=10 (phases 1–4); RepeatedStratifiedKFold 5×10=50 folds (phase 5)  
**AUC Metric:** Median, Q1 (25th), Q3 (75th) across folds  

```
For each CV fold:
    1. Split → training fold (n≈285) and test fold (n≈32)
    2. Scale on training fold only
    3. Compute LRT p-value for each feature on training fold only
    4. Select features with p < threshold
    5. Fit model on selected training features
    6. Evaluate on test fold
```

## Structure
- **Phase 0:** Setup, imports, nested CV engine
- **Phase 1:** Univariate ranking (informational only — not used for selection)
- **Phase 2:** Baseline nested CV across input sets and thresholds
- **Phase 3:** Stability — how many features selected per fold
- **Phase 4:** Hyperparameter sweep (nested CV)
- **Phase 5:** Final verification — 5×10 repeated k-fold

## setup

In [ ]:
# ── Phase 0: Setup ────────────────────────────────────────────────────────
!pip install statsmodels --quiet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
from scipy.stats import chi2 as chi2_dist
import os
import warnings
warnings.filterwarnings('ignore')

TARGET = 'child_face_position_fi___1'
CV10   = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
RSKF   = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)

save_path = # insert save path here.

def auc_stats(scores):
    return np.median(scores), np.percentile(scores, 25), np.percentile(scores, 75)

print("Imports complete.")

Imports complete.


In [ ]:
# ── Nested CV engine ──────────────────────────────────────────────────────
def nested_cv_pval(X_df, y, model_factory, p_threshold=0.1,
                   cv=None, verbose=False):
    if cv is None:
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    cols       = X_df.columns.tolist()
    X_vals     = X_df.values
    y_vals     = y.values

    scores, n_selected = [], []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_vals, y_vals)):
        X_tr, X_val = X_vals[tr_idx], X_vals[val_idx]
        y_tr, y_val = y_vals[tr_idx], y_vals[val_idx]

        # ── Step 1: Scale on training fold only ───────────────────────────
        sc       = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)      # uses training mean/std only

        # ── Step 2: LRT p-value on training fold only ─────────────────────
        selected = []
        for j in range(X_tr_sc.shape[1]):
            X_full = np.hstack([np.ones((len(y_tr), 1)), X_tr_sc[:, j:j+1]])
            try:
                mod_full = sm.Logit(y_tr, X_full).fit(disp=0, method='bfgs')
                mod_null = sm.Logit(y_tr, X_full[:, :1]).fit(disp=0, method='bfgs')
                lrt_stat = 2 * (mod_full.llf - mod_null.llf)
                p_val    = chi2_dist.sf(lrt_stat, df=1)
            except Exception:
                p_val = 1.0
            if p_val < p_threshold:
                selected.append(j)

        if len(selected) == 0:
            scores.append(0.5); n_selected.append(0); continue

        n_selected.append(len(selected))
        if verbose:
            sel_names = [cols[j] for j in selected]
            print(f"    Fold {fold_i+1}: {len(selected)} features selected: {sel_names}")

        # ── Step 3: Fit and evaluate ───────────────────────────────────────
        X_tr_sel  = X_tr_sc[:, selected]
        X_val_sel = X_val_sc[:, selected]
        m         = model_factory()
        m.fit(X_tr_sel, y_tr)
        prob      = m.predict_proba(X_val_sel)[:, 1]
        scores.append(roc_auc_score(y_val, prob))

    return np.array(scores), np.array(n_selected)

def ev_nested(name, model_factory, X_df, y, p_thresh=0.1, cv=None, verbose=False):
    """Run nested CV and print results."""
    scores, n_sel = nested_cv_pval(X_df, y, model_factory,
                                    p_threshold=p_thresh, cv=cv,
                                    verbose=verbose)
    m, q1, q3 = auc_stats(scores)
    n_folds   = len(scores)
    print(f"  {name:<60} Med={m:.4f}  Q1={q1:.4f}  Q3={q3:.4f}  "
          f"[{n_folds} folds, avg {n_sel.mean():.1f} feats/fold]")
    return m, q1, q3, scores, n_sel

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Data/DF_IMPUTED_JUNE.csv'

df    = pd.read_csv(DATA_PATH)
print(f"Preprocessed dataset shape: {df.shape}")

Preprocessed dataset shape: (317, 186)


In [ ]:
df    = df.drop(columns=['sudcrrc_id'], errors='ignore')
X_all = df.drop(columns=[TARGET])
y     = df[TARGET]

print(f"Shape: {df.shape}")
print(f"Analysis Dataset Shape: {X_all.shape}")
print(f"Prone rate: {y.mean():.3f}")

Shape: (317, 185)
Analysis Dataset Shape: (317, 184)
Prone rate: 0.552


## Phase 1: Univariate Ranking

In [ ]:
def run_phase1_univariate(X_all, y, save_path=save_path):

    CATEGORY_MAP = {}

    _pregnancy_hx = [
        'hypertension_during_pregnancy', 'tocolytics_given_during_pregnancy',
        'proteinuria_during_pregnancy', 'bedrest_during_pregnancy',
        'diabetes_req_insulin_during_pregnancy', 'diabetes_not_req_insulin_during_pregnancy',
        'vaginal_bleeding_in_1st_trimester_during_pregnancy',
        'vaginal_bleeding_in_2nd_trimester_during_pregnancy',
        'vaginal_bleeding_in_3rd_trimester_during_pregnancy',
        'cigarette_smoking_during_pregnancy', 'vitamin_during_pregnancy',
        'iron_supplements_during_pregnancy', 'aspirin_during_pregnancy',
        'acetominophen_during_pregnancy', 'ibuprofen_during_pregnancy',
        'antacid_medication_during_pregnancy', 'blood_pressure_medication_during_pregnancy',
        'antidepressants_during_pregnancy', 'decongestants_during_pregnancy',
        'THC_during_pregnancy', 'antibiotics_during_pregnancy',
        'maternal_infection_during_pregnancy', 'gestational_hormones',
        'hormones_during_pregnancy', 'illicit_drug_use_during_pregnancy',
        'premature_labor_history', 'third_trimester_fever',
        'preeclampsia_eclampsia_history', 'any_preg_complications', 'any_alchol_preg',
        'any_caffeine_coffee_tea', 'any_cocaine', 'any_amphetamines', 'any_opioids_heroin',
        'conception_duration_over_12m', 'any_abortions', 'any_miscarriage', 'any_stillbirths',
    ]

    _perinatal_birth = [
        'gestational_week_labor_began', 'spontaneous_labor', 'used_drugs_to_initiate_labor',
        'used_drugs_to_strengthen_contractions', 'child_in_distress_during_labor',
        'meconium_staining_of_amniotic_fluid_present', 'umbilical_cord_problems',
        'child_in_distress_upon_delivery', 'gestational_age_at_birth', 'birthweight_percent',
        'birthlenth_percent', 'apgar_1_min', 'apgar_5_min', 'in_hospital_newborn_anemic',
        'in_hospital_newborn_req_oxygen', 'in_hospital_newborn_intubation',
        'in_hospital_newborn_antibiotics', 'in_hospital_newborn_jaundice',
        'in_hospital_newborn_req_surgery', 'in_hospital_newborn_tachycardia',
        'in_hospital_newborn_apnea', 'in_hospital_newborn_cyanosis',
        'in_hospital_newborn_respiratory_distress', 'in_hospital_newborn_fever',
        'in_hospital_newborn_hypoglycemia', 'in_hospital_newborn_NICU',
        'in_hospital_newborn_discharge_home_w/_mom', 'in_hospital_newborn_extended_hospital_stay',
        'any_delivery_complications', 'if_stop_breathing', 'if_bradycardia', 'if_req_cpr',
        'born_full_term', 'vaginal_birth', 'delivery_require_forceps', 'delivery_require_vacuum',
        'blood_mismatch', 'breastfed_3m_plus', 'newborn_hear_test_results_all_normal',
        'newborn_screen_test_results_all_normal',
    ]

    _maternal_paternal_hx = [
        'mom_educ_level', 'dad_educ_level', 'family_income_fa', 'fertility_meds_IVF',
        'fertility_meds_IUI', 'fertility_meds_injectable', 'fertility_meds_oral',
        'mother_age', 'father_age', 'mother_married', 'any_resident_smokers',
        'any_mother_premie', 'prenatal_care_start_before_2nd_trimester', 'order_preg_sudc',
    ]

    _demographic = [
        'age_sudc', 'female', 'ethnicity_white', 'ethnicity_black', 'ethnicity_hispanic',
        'ethnicity_asian',
    ]

    _developmental = [
        'developmental_milestones_normal', 'expressive_speech_ok', 'comprehension_ok',
        'fine_motor_ok', 'gross_motor_ok', 'behavior_ok', 'rolled_<5m', 'crawled_<10m',
        'started_walking_<17m', 'first_words_<14m', 'general_health_scale',
    ]

    _child_health = [
        'child_autism_MH', 'child_intellectual_disability_MH', 'child_dementia_MH',
        'child_tremor_MH', 'child_asthma_MH', 'child_reactive_airway_disease_MH',
        'child_RSV_MH', 'child_anxiety_MH', 'child_depression_MH', 'child_ADD/ADHD_MH',
        'child_Atrial_Septal_Defect_MH', 'child_Ventricular_Septal_Defect_MH',
        'vaccines_uptodate', 'have_allergy', 'sick_visits_per_year', 'num_er_visits',
        'num_hospital_adm', 'days_illness_inyear', 'sudc_child_h/o_limpness',
        'sudc_child_h/o_blue', 'sudc_child_h/o_choking', 'sudc_child_h/o_>12hr_w/o_eating',
        'sudc_child_h/o_>12hr_vomiting', 'sudc_child_h/o_breathing_problems',
        'sudc_child_h/o_heart_problems', 'sudc_child_h/o_GER', 'sudc_child_use_apnea_monitor',
        'sudc_child_hearing_impairment', 'sudc_child_speech_delay',
        'sudc_child_vision_impairment', 'sudc_child_h/o_wheezing', 'sudc_child_h/o_fainting',
    ]

    _child_health_sz = [
        'febrile_sz___any_type_Final', 'if_have_seizure', 'simple_fs', 'complex_fs',
    ]

    _prodromal_acute = [
        'recent_head_trauma', 'recent_flying_on_airplane', 'recent_visit_ER',
        'recent_visit_doctor', 'recent_exposure_contagious_disease', 'recent_suffer_illness',
        'last_48hr_cold_symptoms', 'last_48hr_lethargy', 'last_48hr_crankiness',
        'last_48hr_excessive_crying', 'last_48hr_appetite_changes', 'last_48hr_vomiting',
        'last_48hr_fever', 'last_48hr_diarrhea', 'last_48hr_other_stool_changes',
    ]

    _sleep_behavior = [
        'talked_during_sleep_<6m_prior', 'restless_during_sleep_<6m_prior',
        'sleepwalked_<6m_prior', 'grinded_teeth_<6m_prior', 'night_terror_<6m_prior',
        'awoke_sweating_<6m_prior', 'scary_dreams_<6m_prior', 'awoke_often_>2/week',
        'snoring', 'difficulty_falling_asleep', 'difficulty_staying_asleep',
        'have_favorite_sleep_position',
    ]

    _circumstances = [
        'any_bed_sharing', 'body_position_prone', 'body_position_supine', 'body_position_side',
    ]

    _family_history = [
        'any_fam_FS', 'any_fam_cardiac_arrhythmia', 'any_fam_syncope', 'any_fam_autism',
        'any_fam_sudden_cardiac_death', 'any_fam_intellectual_disabilities',
        'any_fam_other_neurological', 'any_fam_epilepsy', 'any_fam_sleep_apnea',
        'any_fam_asthma', 'any_fam_sudden_explained_death', 'any_fam_sudden_unexplained_death',
    ]

    for var_list, cat_name in [
        (_pregnancy_hx, 'pregnancy_hx'), (_perinatal_birth, 'perinatal_birth'),
        (_maternal_paternal_hx, 'maternal_paternal_hx'), (_demographic, 'demographic'),
        (_developmental, 'developmental'), (_child_health, 'child_health'),
        (_child_health_sz, 'child_health_sz'), (_prodromal_acute, 'prodromal_acute'),
        (_sleep_behavior, 'sleep_behavior'), (_circumstances, 'circumstances'),
        (_family_history, 'family_history'),
    ]:
        for v in var_list:
            CATEGORY_MAP[v] = cat_name

    def categorize(f):
        # Exact match first; fall back to 'unclassified' (not 'other') so any
        # newly-added variable not yet placed in CATEGORY_MAP is impossible to miss.
        return CATEGORY_MAP.get(f, 'unclassified')

    def get_counts(series):
        unique_vals = set(series.dropna().unique())
        if unique_vals.issubset({0, 1, 0.0, 1.0}):
            return int((series == 1).sum()), int((series == 0).sum())
        return 'NA', 'NA'

    def get_feature_type(series):
        unique_vals = sorted(series.dropna().unique())
        n_unique = len(unique_vals)
        if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
            return 'binary', None
        elif n_unique <= 10 and all(float(v).is_integer() for v in unique_vals):
            return 'categorical', n_unique
        else:
            return 'continuous', None

    features = X_all.columns.tolist()
    print(f"{len(features)} features tested")

    unclassified = [f for f in features if categorize(f) == 'unclassified']
    if unclassified:
        print(f"\n[WARNING] {len(unclassified)} features have no category assignment:")
        for f in unclassified:
            print(f"    '{f}'")

    rows = []
    for feat in features:
        X_feat = X_all[[feat]].values
        pipe   = Pipeline([('scaler', StandardScaler()),
                           ('lr', LogisticRegression(C=1.0, max_iter=1000, random_state=42))])
        aucs   = cross_val_score(pipe, X_feat, y, cv=CV10, scoring='roc_auc')
        coef_pipe = Pipeline([('scaler', StandardScaler()),
                              ('lr', LogisticRegression(C=1.0, max_iter=1000, random_state=42))])
        coef_pipe.fit(X_feat, y)
        direction = '+' if coef_pipe.named_steps['lr'].coef_[0][0] > 0 else '-'
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(X_feat)
        X_full   = sm.add_constant(X_scaled)
        try:
            mod_full = sm.Logit(y, X_full).fit(disp=0, method='bfgs')
            mod_null = sm.Logit(y, X_full[:, :1]).fit(disp=0, method='bfgs')
            lrt_stat = 2 * (mod_full.llf - mod_null.llf)
            p_value  = chi2_dist.sf(lrt_stat, df=1)
        except Exception:
            p_value  = 1.0
        pos_count, neg_count = get_counts(X_all[feat])
        feat_type, n_cats    = get_feature_type(X_all[feat])
        rows.append({'feature': feat, 'category': categorize(feat),
                     'feat_type': feat_type, 'n_categories': n_cats,
                     'pos_count': pos_count, 'neg_count': neg_count,
                     'median_auc': np.median(aucs), 'q1_auc': np.percentile(aucs, 25),
                     'q3_auc': np.percentile(aucs, 75), 'direction': direction,
                     'lrt_p_value': p_value})

    # --- Build ranked DataFrames ---
    results_df      = pd.DataFrame(rows)
    results_df_full = results_df.sort_values('median_auc', ascending=False).copy()
    results_df_full['auc_rank'] = range(1, len(results_df_full) + 1)
    results_df_sig  = results_df[results_df['lrt_p_value'] < 0.1].sort_values('median_auc', ascending=False).copy()
    results_df_sig['auc_rank']  = range(1, len(results_df_sig) + 1)

    # --- Print ---
    print(f"\nPHASE 1: UNIVARIATE RANKING (INFORMATIONAL — computed on full dataset)")
    print(f"n={len(y)}, prone_rate={y.mean():.3f}")
    print(f"Features with full-data p<0.1: {len(results_df_sig)} of {len(features)}\n")
    print(f"TOP 25 BY MEDIAN AUC:")
    print(f"{'Rank':<5} {'Feature':<40} {'Category':<22} {'Type':<12} {'NCat':>5} {'Dir':<4} {'Pos':>6} {'Neg':>6} {'Median':>8} {'Q1':>8} {'Q3':>8} {'p-value':>10} {'Sig'}")
    print("-" * 149)
    for _, row in results_df_full.head(25).iterrows():
        sig = '★ p<0.05' if row['lrt_p_value'] < 0.05 else ('† p<0.10' if row['lrt_p_value'] < 0.10 else '')
        n_cat_str = str(int(row['n_categories'])) if pd.notna(row['n_categories']) else '—'
        print(f"{row['auc_rank']:<5} {row['feature']:<40} {row['category']:<22} {row['feat_type']:<12} {n_cat_str:>5} {row['direction']:<4} "
              f"{str(row['pos_count']):>6} {str(row['neg_count']):>6} "
              f"{row['median_auc']:>8.4f} {row['q1_auc']:>8.4f} {row['q3_auc']:>8.4f} "
              f"{row['lrt_p_value']:>10.4f}  {sig}")

    print(f"\nCATEGORY SUMMARY (p<0.1):")
    cat_summary = results_df_sig.groupby('category').agg(
        n_sig=('feature','count'), mean_auc=('median_auc','mean'),
        max_auc=('median_auc','max'), best_feature=('feature','first'),
        min_p=('lrt_p_value','min')
    ).sort_values('max_auc', ascending=False)
    print(cat_summary.to_string())

    # --- Save ---
    results_df_full.to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase1_data_detailed.csv'),
        index=False
    )
    cat_summary.reset_index().to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase1_data_summary.csv'),
        index=False
    )
    return results_df_full, results_df_sig, cat_summary

In [ ]:
results_df_full, results_df_sig, cat_symmary = run_phase1_univariate(X_all, y)

184 features tested

PHASE 1: UNIVARIATE RANKING (INFORMATIONAL — computed on full dataset)
n=317, prone_rate=0.552
Features with full-data p<0.1: 31 of 184

TOP 25 BY MEDIAN AUC:
Rank  Feature                                  Category               Type          NCat Dir     Pos    Neg   Median       Q1       Q3    p-value Sig
-----------------------------------------------------------------------------------------------------------------------------------------------------
1     body_position_prone                      circumstances          binary           — +       260     57   0.6746   0.6161   0.7007     0.0000  ★ p<0.05
2     body_position_supine                     circumstances          binary           — -        30    287   0.5877   0.5444   0.6071     0.0000  ★ p<0.05
3     body_position_side                       circumstances          binary           — -        29    288   0.5857   0.5714   0.6131     0.0000  ★ p<0.05
4     dad_educ_level                           mater

## Phase 2: Baseline Nested CV

Three **candidate input sets** are fed into nested CV. The p-value filter runs inside each fold.

- **Clinical (12f):** Domain-motivated features known to matter clinically
- **Broad (34f):** Clinical + top features from Phase 1 univariate ranking
- **All (220f):** All features — selection does all the work inside each fold

Two thresholds tested: p<0.1 and p<0.05.

maybe try only keeping the ones under 0.1 as the narrow set, and the broad set be narrow set + clinical features that were missed.

In [ ]:
lr_C0_001 = lambda: LogisticRegression(C=0.001, penalty='l2', max_iter=2000, random_state=42)
lr_C0_01  = lambda: LogisticRegression(C=0.01, penalty='l2', max_iter=2000, random_state=42)
lr_C0_1   = lambda: LogisticRegression(C=0.1,  penalty='l2', max_iter=2000, random_state=42)
lr_C1_0   = lambda: LogisticRegression(C=1.0,  penalty='l2', max_iter=2000, random_state=42)
rf_d3     = lambda: RandomForestClassifier(n_estimators=100, max_depth=3, max_features='sqrt', random_state=42)
rf_d5     = lambda: RandomForestClassifier(n_estimators=100, max_depth=5, max_features='sqrt', random_state=42)


In [ ]:
def run_phase2_nested_cv(X_all=X_all, y=y, save_path=save_path, verbose_diagnostics=True):

    # ── 0. Sanity check: confirm X_all is not pre-filtered ──────────────────
    if verbose_diagnostics:
        print(f"[DIAGNOSTIC] X_all shape going into Phase 2: {X_all.shape}")
        print(f"[DIAGNOSTIC] Confirm this equals your full preprocessed feature count "
              f"(184 expected as of the 2026-07 preprocessing fix -- snoring and "
              f"blood_mismatch were dropped, so this is 184, not the earlier 186). "
              f"If this number doesn't match, X_all has already been filtered upstream "
              f"of this function and every fold below will leak full-sample information "
              f"through that pre-filter, regardless of how clean the code inside "
              f"nested_cv_pval is.\n")

    # --- Reference (no p-value filter, all features) ---
    ref_scores   = cross_val_score(
        Pipeline([('sc', StandardScaler()), ('m', lr_C0_01())]),
        X_all, y, cv=CV10, scoring='roc_auc')
    rm, rq1, rq3 = auc_stats(ref_scores)
    print(f"REFERENCE (no p-value filter, all {X_all.shape[1]}f): "
          f"Med={rm:.4f}  Q1={rq1:.4f}  Q3={rq3:.4f}\n")

    # ── 1. Permutation sanity check (shuffled y) ─────────────────────────────
    print("="*70)
    print("PERMUTATION SANITY CHECK (y shuffled -- expect AUC ~0.5 everywhere)")
    print("="*70)
    rng = np.random.RandomState(42)
    y_shuffled = pd.Series(rng.permutation(y.values), index=y.index)

    for thresh in [0.1, 0.05]:
        scores_shuf, n_sel_shuf = nested_cv_pval(
            X_all, y_shuffled, lr_C0_01, p_threshold=thresh, cv=CV5)
        m_shuf, q1_shuf, q3_shuf = auc_stats(scores_shuf)
        flag = "  [!!! INVESTIGATE — not ~0.5 !!!]" if abs(m_shuf - 0.5) > 0.07 else "  [OK]"
        print(f"  p<{thresh}  shuffled-y  LR_C0.01  Med={m_shuf:.4f}  "
              f"Q1={q1_shuf:.4f}  Q3={q3_shuf:.4f}  avg_feats={n_sel_shuf.mean():.1f}{flag}")
    print()

    # --- Nested CV across both p-value thresholds and all models ---
    print("PHASE 2: NESTED CV RESULTS (real y)")
    print(f"  {'Config':<45} {'Median':>7} {'Q1':>7} {'Q3':>7}  avg_feats")
    print(f"  {'-'*80}")

    detailed_rows = []
    summary_rows  = []
    selected_features_by_config = {}   # for stability diagnostic below

    for thresh in [0.1, 0.05]:
        for mname, mfac in [('LR_C0.01', lr_C0_01), ('LR_C0.1', lr_C0_1),
                              ('LR_C1.0', lr_C1_0),  ('RF_d3', rf_d3),
                              ('RF_d5', rf_d5)]:
            scores, n_sel, fold_features = nested_cv_pval_with_names(
                X_all, y, mfac, p_threshold=thresh, cv=CV10)
            m, q1, q3 = auc_stats(scores)
            label = f"p<{thresh} {mname}"
            print(f"  {label:<45} {m:>7.4f} {q1:>7.4f} {q3:>7.4f}  {n_sel.mean():.1f}")

            for fold_i, (auc_i, nsel_i) in enumerate(zip(scores, n_sel)):
                detailed_rows.append({
                    'p_threshold': thresh, 'model': mname, 'fold': fold_i + 1,
                    'auc': auc_i, 'n_feats_selected': nsel_i,
                })
            summary_rows.append({
                'p_threshold': thresh, 'model': mname,
                'median_auc': m, 'q1_auc': q1, 'q3_auc': q3,
                'avg_feats_selected': n_sel.mean(), 'std_feats_selected': n_sel.std(),
            })

            # Only need feature-stability check once per threshold, not per model
            # (selection doesn't depend on which downstream model is used)
            if mname == 'LR_C0.01':
                selected_features_by_config[thresh] = fold_features

    # ── 2. Feature selection stability across folds ─────────────────────────
    print("\n" + "="*70)
    print("FEATURE SELECTION STABILITY ACROSS FOLDS")
    print("="*70)
    stable_features = {}   # thresh -> list of features selected in EVERY fold
    for thresh, fold_features in selected_features_by_config.items():
        print(f"\np<{thresh}:")
        all_feats = set()
        for f in fold_features:
            all_feats.update(f)
        print(f"  Union of features selected in ANY fold: {len(all_feats)}")

        from collections import Counter
        feat_counts = Counter()
        for f in fold_features:
            feat_counts.update(f)
        n_folds = len(fold_features)
        always_selected = [f for f, c in feat_counts.items() if c == n_folds]
        rarely_selected = [f for f, c in feat_counts.items() if c <= 2]
        stable_features[thresh] = sorted(always_selected)
        print(f"  Features selected in ALL {n_folds} folds: {len(always_selected)}")
        print(f"    {sorted(always_selected)}")
        print(f"  Features selected in <=2 folds (likely fold-specific noise): "
              f"{len(rarely_selected)}")
        print(f"    {sorted(rarely_selected)}")

        jaccards = []
        for i in range(len(fold_features)):
            for j in range(i+1, len(fold_features)):
                a, b = set(fold_features[i]), set(fold_features[j])
                if len(a | b) > 0:
                    jaccards.append(len(a & b) / len(a | b))
        print(f"  Mean pairwise Jaccard overlap between folds: {np.mean(jaccards):.3f} "
              f"(1.0 = identical sets every fold, 0.0 = no overlap)")

    # Persist stable-feature lists so downstream phases (3, 4, 5) can consume
    # them dynamically instead of re-deriving or hardcoding a copy.
    stable_rows = []
    for thresh, feats in stable_features.items():
        for f in feats:
            stable_rows.append({'p_threshold': thresh, 'feature': f})
    pd.DataFrame(stable_rows).to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase2_stable_features.csv'),
        index=False)

    # --- Top 10 ---
    summary_df = pd.DataFrame(summary_rows).sort_values('median_auc', ascending=False)
    print(f"\nTOP 10 CONFIGURATIONS:")
    print(f"  {'thresh':>6} {'Model':<12} {'Median':>8} {'Q1':>8} {'Q3':>8}  avg_feats")
    print(f"  {'-'*55}")
    for _, r in summary_df.head(10).iterrows():
        print(f"  {r['p_threshold']:>6.2f} {r['model']:<12} "
              f"{r['median_auc']:>8.4f} {r['q1_auc']:>8.4f} {r['q3_auc']:>8.4f}  "
              f"{r['avg_feats_selected']:.1f}")

    ref_row = pd.DataFrame([{
        'p_threshold': 'none', 'model': 'LR_C0.01_reference',
        'median_auc': rm, 'q1_auc': rq1, 'q3_auc': rq3,
        'avg_feats_selected': X_all.shape[1], 'std_feats_selected': 0,
    }])

    pd.DataFrame(detailed_rows).to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase2_data_detailed.csv'),
        index=False)
    pd.concat([summary_df, ref_row], ignore_index=True).to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase2_data_summary.csv'),
        index=False)

    return pd.DataFrame(detailed_rows), summary_df, stable_features


def nested_cv_pval_with_names(X_df, y, model_factory, p_threshold=0.1, cv=None):
    """
    Same as nested_cv_pval, but also returns the list of selected feature
    NAMES per fold (not just the count), needed for the stability diagnostic.
    """
    if cv is None:
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    cols   = X_df.columns.tolist()
    X_vals = X_df.values
    y_vals = y.values

    scores, n_selected, fold_feature_names = [], [], []

    for tr_idx, val_idx in cv.split(X_vals, y_vals):
        X_tr, X_val = X_vals[tr_idx], X_vals[val_idx]
        y_tr, y_val = y_vals[tr_idx], y_vals[val_idx]

        sc       = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)

        selected = []
        for j in range(X_tr_sc.shape[1]):
            X_full = np.hstack([np.ones((len(y_tr), 1)), X_tr_sc[:, j:j+1]])
            try:
                mod_full = sm.Logit(y_tr, X_full).fit(disp=0, method='bfgs')
                mod_null = sm.Logit(y_tr, X_full[:, :1]).fit(disp=0, method='bfgs')
                lrt_stat = 2 * (mod_full.llf - mod_null.llf)
                p_val    = chi2_dist.sf(lrt_stat, df=1)
            except Exception:
                p_val = 1.0
            if p_val < p_threshold:
                selected.append(j)

        fold_feature_names.append([cols[j] for j in selected])

        if len(selected) == 0:
            scores.append(0.5); n_selected.append(0); continue

        n_selected.append(len(selected))
        X_tr_sel  = X_tr_sc[:, selected]
        X_val_sel = X_val_sc[:, selected]
        m = model_factory()
        m.fit(X_tr_sel, y_tr)
        prob = m.predict_proba(X_val_sel)[:, 1]
        scores.append(roc_auc_score(y_val, prob))

    return np.array(scores), np.array(n_selected), fold_feature_names

In [ ]:
phase2_detailed, phase2_summary, stable_features = run_phase2_nested_cv()


[DIAGNOSTIC] X_all shape going into Phase 2: (317, 184)
[DIAGNOSTIC] Confirm this equals your full preprocessed feature count (184 expected as of the 2026-07 preprocessing fix -- snoring and blood_mismatch were dropped, so this is 184, not the earlier 186). If this number doesn't match, X_all has already been filtered upstream of this function and every fold below will leak full-sample information through that pre-filter, regardless of how clean the code inside nested_cv_pval is.

REFERENCE (no p-value filter, all 184f): Med=0.6878  Q1=0.5903  Q3=0.7462

PERMUTATION SANITY CHECK (y shuffled -- expect AUC ~0.5 everywhere)
  p<0.1  shuffled-y  LR_C0.01  Med=0.4857  Q1=0.4577  Q3=0.5187  avg_feats=20.2  [OK]
  p<0.05  shuffled-y  LR_C0.01  Med=0.4635  Q1=0.4602  Q3=0.4735  avg_feats=9.8  [OK]

PHASE 2: NESTED CV RESULTS (real y)
  Config                                         Median      Q1      Q3  avg_feats
  -----------------------------------------------------------------------------

## Phase 3: Feature Selection Stability

How many features are selected per fold, and which ones appear consistently?

In [ ]:
def get_low_prevalence_features(X_df, min_count=5):
    """
    Binary features with fewer than min_count cases in the minority class.
    These can produce artificially significant LRT p-values via near-complete
    separation within a ~285-subject training fold.
    Flagged dynamically rather than hardcoding a name list,
    so any future sparse feature gets caught the same way.
    """
    flagged = []
    for c in X_df.columns:
        vals = set(X_df[c].dropna().unique())
        if vals.issubset({0, 1, 0.0, 1.0}):
            pos = int((X_df[c] == 1).sum())
            neg = int((X_df[c] == 0).sum())
            if min(pos, neg) < min_count:
                flagged.append(c)
    return flagged


def run_phase3_ablation(X_all=X_all, y=y, save_path=save_path,
                         base_features=None, model_factory=None, cv=None):
    """
    Systematic feature ablation (SET_CLINICAL/SET_BROAD retired 2026-07).
    Starts from a FIXED feature set (the Phase 2 stable core, by default) --
    no per-fold p-value re-selection here, since these features already
    passed that bar. For each feature, remove it and re-score the remaining
    set on the SAME cv splits as the full-set baseline (paired comparison).

    A feature whose removal DROPS the median AUC is pulling its weight.
    A feature whose removal doesn't change (or improves) the median AUC is
    being carried passively -- a candidate to drop from the clinician-facing
    feature set.
    """
    if base_features is None:
        raise ValueError("base_features must be provided explicitly (e.g. "
                          "the Phase 2 stable-feature list) -- no implicit "
                          "default so it's always clear what's being ablated.")
    if model_factory is None:
        model_factory = lr_C0_01
    if cv is None:
        cv = CV10

    base_features = list(base_features)
    X_base = X_all[base_features]

    full_scores = cross_val_score(
        Pipeline([('sc', StandardScaler()), ('m', model_factory())]),
        X_base, y, cv=cv, scoring='roc_auc')
    full_m, full_q1, full_q3 = auc_stats(full_scores)

    print("PHASE 3: FEATURE ABLATION")
    print(f"Base set: {len(base_features)} features")
    print(f"  {base_features}\n")
    print(f"FULL SET  Med={full_m:.4f}  Q1={full_q1:.4f}  Q3={full_q3:.4f}\n")
    print(f"{'Removed feature':<40} {'n_feat':>7} {'Median':>8} {'Q1':>8} {'Q3':>8}  {'ΔMedian':>9}")
    print("-" * 90)

    rows = [{
        'removed_feature': '(none - full set)', 'n_features': len(base_features),
        'median_auc': full_m, 'q1_auc': full_q1, 'q3_auc': full_q3,
        'delta_median_auc': 0.0,
    }]

    for feat in base_features:
        remaining = [f for f in base_features if f != feat]
        scores = cross_val_score(
            Pipeline([('sc', StandardScaler()), ('m', model_factory())]),
            X_all[remaining], y, cv=cv, scoring='roc_auc')
        m, q1, q3 = auc_stats(scores)
        delta = m - full_m
        print(f"{feat:<40} {len(remaining):>7} {m:>8.4f} {q1:>8.4f} {q3:>8.4f}  {delta:>+9.4f}")
        rows.append({
            'removed_feature': feat, 'n_features': len(remaining),
            'median_auc': m, 'q1_auc': q1, 'q3_auc': q3,
            'delta_median_auc': delta,
        })

    ablation_df = pd.DataFrame(rows).sort_values('delta_median_auc')

    print(f"\nFeatures whose removal HURT most (most valuable, drop worsens AUC):")
    for _, r in ablation_df[ablation_df['removed_feature'] != '(none - full set)'].head(5).iterrows():
        print(f"  {r['removed_feature']:<40} ΔMedian={r['delta_median_auc']:+.4f}")

    print(f"\nFeatures whose removal HELPED or DIDN'T MATTER (candidates to drop):")
    candidates = ablation_df[
        (ablation_df['removed_feature'] != '(none - full set)') &
        (ablation_df['delta_median_auc'] >= 0)
    ].sort_values('delta_median_auc', ascending=False)
    for _, r in candidates.iterrows():
        print(f"  {r['removed_feature']:<40} ΔMedian={r['delta_median_auc']:+.4f}")
    if len(candidates) == 0:
        print("  (none -- every feature's removal hurt performance)")

    ablation_df.to_csv(os.path.join(save_path, 'interpretable_modeling_phase3_ablation.csv'),
                        index=False)
    return ablation_df


In [ ]:
# Base set: Phase 2's p<0.1 stable core (15 features, selected in every fold),
# minus any low-prevalence feature that could be a separation artifact rather
# than real signal (see get_low_prevalence_features docstring above).
_base_p01 = stable_features[0.1]
_low_prev = get_low_prevalence_features(X_all[_base_p01])
if _low_prev:
    print(f"Excluding {len(_low_prev)} low-prevalence feature(s) from ablation base set: {_low_prev}\n")
_ablation_base = [f for f in _base_p01 if f not in _low_prev]

ablation_df = run_phase3_ablation(base_features=_ablation_base, model_factory=lr_C0_01, cv=CV10)


Excluding 2 low-prevalence feature(s) from ablation base set: ['child_Atrial_Septal_Defect_MH', 'in_hospital_newborn_cyanosis']

PHASE 3: FEATURE ABLATION
Base set: 13 features
  ['age_sudc', 'any_bed_sharing', 'behavior_ok', 'body_position_prone', 'body_position_side', 'body_position_supine', 'cigarette_smoking_during_pregnancy', 'febrile_sz___any_type_Final', 'gross_motor_ok', 'hormones_during_pregnancy', 'last_48hr_crankiness', 'last_48hr_fever', 'mother_married']

FULL SET  Med=0.7778  Q1=0.7023  Q3=0.8036

Removed feature                           n_feat   Median       Q1       Q3    ΔMedian
------------------------------------------------------------------------------------------
age_sudc                                      12   0.7778   0.6977   0.7936    +0.0000
any_bed_sharing                               12   0.7791   0.7012   0.8015    +0.0013
behavior_ok                                   12   0.7738   0.6953   0.7812    -0.0040
body_position_prone                         

## Phase 4: Hyperparameter Sweep (Nested CV)

In [ ]:
def run_phase4_hyperparam_sweep(X_all=X_all, y=y, save_path=save_path):
    """SET_CLINICAL/SET_BROAD retired 2026-07 -- sweeps on the full 184-feature
    set with per-fold p-value selection (same as Phase 2), across a broader
    hyperparameter grid than Phase 2 used."""

    model_factories = [
        ('LR_C0.001', lambda: LogisticRegression(C=0.001, max_iter=2000, random_state=42)),
        ('LR_C0.01',  lambda: LogisticRegression(C=0.01,  max_iter=2000, random_state=42)),
        ('LR_C0.1',   lambda: LogisticRegression(C=0.1,   max_iter=2000, random_state=42)),
        ('LR_C1.0',   lambda: LogisticRegression(C=1.0,   max_iter=2000, random_state=42)),
        ('RF_n100d3', lambda: RandomForestClassifier(n_estimators=100, max_depth=3, max_features='sqrt', random_state=42)),
        ('RF_n100d5', lambda: RandomForestClassifier(n_estimators=100, max_depth=5, max_features='sqrt', random_state=42)),
        ('RF_n200d3', lambda: RandomForestClassifier(n_estimators=200, max_depth=3, max_features='sqrt', random_state=42)),
        ('RF_n200d5', lambda: RandomForestClassifier(n_estimators=200, max_depth=5, max_features='sqrt', random_state=42)),
    ]

    print("PHASE 4: HYPERPARAMETER SWEEP (nested CV, all 184 features, per-fold p-value selection)")

    detailed_rows = []
    summary_rows  = []

    for thresh in [0.1, 0.05]:
        print(f"\n── p<{thresh} ──")
        print(f"  {'Model':<14} {'Median':>8} {'Q1':>8} {'Q3':>8}  avg_feats  std_auc")
        print(f"  {'-'*60}")

        for mname, mfac in model_factories:
            scores, n_sel = nested_cv_pval(
                X_all, y, mfac, p_threshold=thresh, cv=CV10)
            m, q1, q3 = auc_stats(scores)
            auc_std = np.std(scores)
            auc_min = np.min(scores)
            auc_max = np.max(scores)

            print(f"  {mname:<14} {m:>8.4f} {q1:>8.4f} {q3:>8.4f}  "
                  f"{n_sel.mean():.1f}      {auc_std:.4f}")

            for fold_i, (auc_i, nsel_i) in enumerate(zip(scores, n_sel)):
                detailed_rows.append({
                    'p_threshold': thresh, 'model': mname, 'fold': fold_i + 1,
                    'auc': auc_i, 'n_feats_selected': nsel_i,
                })
            summary_rows.append({
                'p_threshold': thresh, 'model': mname,
                'median_auc': m, 'q1_auc': q1, 'q3_auc': q3,
                'mean_auc': np.mean(scores), 'std_auc': auc_std,
                'min_auc': auc_min, 'max_auc': auc_max, 'auc_range': auc_max - auc_min,
                'avg_feats_selected': n_sel.mean(), 'std_feats_selected': n_sel.std(),
                'min_feats_selected': n_sel.min(), 'max_feats_selected': n_sel.max(),
            })

    summary_df = pd.DataFrame(summary_rows).sort_values('median_auc', ascending=False)
    print(f"\n\nTOP 15 CONFIGURATIONS OVERALL:")
    print(f"  {'thresh':>6} {'Model':<12} {'Median':>8} {'Q1':>8} {'Q3':>8}  avg_feats  std_auc  auc_range")
    print(f"  {'-'*80}")
    for _, r in summary_df.head(15).iterrows():
        print(f"  {r['p_threshold']:>6.2f} {r['model']:<12} "
              f"{r['median_auc']:>8.4f} {r['q1_auc']:>8.4f} {r['q3_auc']:>8.4f}  "
              f"{r['avg_feats_selected']:.1f}       {r['std_auc']:.4f}   {r['auc_range']:.4f}")

    pd.DataFrame(detailed_rows).to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase4_data_detailed.csv'), index=False)
    summary_df.to_csv(
        os.path.join(save_path, 'interpretable_modeling_phase4_data_summary.csv'), index=False)

    return pd.DataFrame(detailed_rows), summary_df


In [ ]:
phase4_detailed, phase4_summary = run_phase4_hyperparam_sweep()


PHASE 4: HYPERPARAMETER SWEEP (nested CV, all 184 features, per-fold p-value selection)

── p<0.1 ──
  Model            Median       Q1       Q3  avg_feats  std_auc
  ------------------------------------------------------------
  LR_C0.001        0.7482   0.6210   0.7599  31.5      0.0880
  LR_C0.01         0.7269   0.6022   0.7561  31.5      0.0940
  LR_C0.1          0.7049   0.5794   0.7447  31.5      0.1010
  LR_C1.0          0.6966   0.5764   0.7325  31.5      0.0980
  RF_n100d3        0.6551   0.6469   0.7610  31.5      0.0738
  RF_n100d5        0.7003   0.6435   0.7460  31.5      0.0701
  RF_n200d3        0.6667   0.6329   0.7498  31.5      0.0698
  RF_n200d5        0.6917   0.6624   0.7468  31.5      0.0644

── p<0.05 ──
  Model            Median       Q1       Q3  avg_feats  std_auc
  ------------------------------------------------------------
  LR_C0.001        0.7024   0.6607   0.7477  21.0      0.0694
  LR_C0.01         0.6899   0.6311   0.7445  21.0      0.0700
  LR_C0.1  

## Phase 5: Final Verification — 5×10 Repeated K-Fold

In [ ]:
from collections import Counter

In [ ]:
def run_phase5_final_verification(X_all=X_all, y=y, save_path=save_path):

    champion_configs = [
    ("Nested p<0.1  LR_C0.01,  all184",
     lr_C0_01, X_all, 0.1),

    ("Nested p<0.1  LR_C0.001, all184",
     lr_C0_001, X_all, 0.1),

    ("Nested p<0.05 LR_C0.001, all184",
     lr_C0_001, X_all, 0.05),

    ("Nested p<0.1  RF_d3,     all184",
     rf_d3, X_all, 0.1),

    ("Nested p<0.05 RF_d3,     all184",
     rf_d3, X_all, 0.05),

    ("Nested p<0.05 LR_C0.01,  all184",
     lr_C0_01, X_all, 0.05),

    ("REFERENCE: no filter, LR_C0.01, all184",
     None, X_all, None),
    ]

    print("PHASE 5: FINAL VERIFICATION — 5×10 REPEATED K-FOLD (50 folds)\n")
    print(f"  {'Config':<45} {'Med':>8} {'Q1':>8} {'Q3':>8}  IQR  avg_feats")
    print(f"  {'-'*92}")

    all_scores    = {}
    all_n_sel     = {}
    detailed_rows = []
    summary_rows  = []
    best_m        = -1
    CHAMPION_NAME = ""

    for name, mfac, X_df, thresh in champion_configs:
        if thresh is None:
            scores = cross_val_score(
                Pipeline([('sc', StandardScaler()),
                          ('m', LogisticRegression(C=0.01, max_iter=2000, random_state=42))]),
                X_df, y, cv=RSKF, scoring='roc_auc')
            n_sel     = np.array([X_df.shape[1]] * len(scores))
            n_sel_str = f"{X_df.shape[1]:.1f} (fixed)"
        else:
            scores, n_sel = nested_cv_pval(X_df, y, mfac, p_threshold=thresh, cv=RSKF)
            n_sel_str = f"{n_sel.mean():.1f}"

        all_scores[name] = scores
        all_n_sel[name]  = n_sel
        m, q1, q3        = auc_stats(scores)

        if m > best_m:
            best_m = m
            CHAMPION_NAME = name

        print(f"  {name:<45} {m:>8.4f} {q1:>8.4f} {q3:>8.4f}  {q3-q1:.4f}  {n_sel_str}")

    for name, scores in all_scores.items():
        n_sel = all_n_sel[name]
        m, q1, q3 = auc_stats(scores)
        thresh = next(c[3] for c in champion_configs if c[0] == name)

        for fold_i, (auc_i, nsel_i) in enumerate(zip(scores, n_sel)):
            detailed_rows.append({
                'config': name, 'p_threshold': thresh, 'fold': fold_i + 1,
                'auc': auc_i, 'n_feats_selected': nsel_i,
                'is_champion': name == CHAMPION_NAME, 'is_reference': thresh is None,
            })
        summary_rows.append({
            'config': name, 'p_threshold': thresh, 'n_folds': len(scores),
            'median_auc': m, 'q1_auc': q1, 'q3_auc': q3, 'iqr_auc': q3 - q1,
            'mean_auc': np.mean(scores), 'std_auc': np.std(scores),
            'is_champion': name == CHAMPION_NAME, 'is_reference': thresh is None,
        })

    print(f"\nWINNING CONFIG: {CHAMPION_NAME} (Median AUC: {best_m:.4f})")

    print(f"\nCHAMPION ANALYTICS: Feature frequency across 10 folds")
    champ_X_df   = next(c[2] for c in champion_configs if c[0] == CHAMPION_NAME)
    champ_thresh = next(c[3] for c in champion_configs if c[0] == CHAMPION_NAME)

    if champ_thresh is not None:
        cols       = champ_X_df.columns.tolist()
        X_vals     = champ_X_df.values
        y_vals     = y.values
        CV10_fresh = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

        freq = Counter()
        for tr_idx, _ in CV10_fresh.split(X_vals, y_vals):
            X_tr, y_tr = X_vals[tr_idx], y_vals[tr_idx]
            X_tr_sc = StandardScaler().fit_transform(X_tr)
            fold_selected = set()

            for j in range(X_tr_sc.shape[1]):
                X_full = sm.add_constant(X_tr_sc[:, j])
                try:
                    mod_full = sm.Logit(y_tr, X_full).fit(disp=0, method='bfgs')
                    mod_null = sm.Logit(y_tr, X_full[:, :1]).fit(disp=0, method='bfgs')
                    p_val    = chi2_dist.sf(2 * (mod_full.llf - mod_null.llf), df=1)  # FIXED: was bare `chi2` (undefined -> silent 1.0 via bare except)
                except Exception:
                    p_val = 1.0

                if p_val < champ_thresh:
                    fold_selected.add(cols[j])

            for feat in fold_selected:
                freq[feat] += 1

        print(f"{'Feature':<40} {'Folds':>10} {'%':>6}")
        print("-" * 60)
        freq_rows = []
        for feat, count in sorted(freq.items(), key=lambda x: (-x[1], x[0])):
            print(f"{feat:<40} {count:>10}/10  {count*10:>5}%")
            freq_rows.append({'feature': feat, 'folds_selected': count, 'pct_selected': count*10})

        low_prev_in_champ = get_low_prevalence_features(champ_X_df, min_count=5)
        high_freq_low_prev = [r['feature'] for r in freq_rows
                               if r['feature'] in low_prev_in_champ and r['pct_selected'] >= 70]
        if high_freq_low_prev:
            print(f"\n[CAVEAT] {len(high_freq_low_prev)} feature(s) selected in >=70% of folds "
                  f"have fewer than 5 cases in the minority class: {high_freq_low_prev}")
            print( "  With ~285 training subjects per fold, a feature this sparse can achieve "
                   "artificially low LRT p-values via near-complete separation rather than "
                   "reflecting real signal. Retained here rather than dropped, but should be "
                   "reported with this caveat rather than presented as equivalent in strength "
                   "to the higher-prevalence stable features (e.g. febrile_sz___any_type_Final, "
                   "body_position_*).")
    else:
        print("Champion is a reference model; no feature selection to analyze.")
        freq_rows = []

    auc_50_honest = all_scores[CHAMPION_NAME]

    pd.DataFrame(detailed_rows).to_csv(os.path.join(save_path, 'phase5_detailed.csv'), index=False)
    pd.DataFrame(summary_rows).to_csv(os.path.join(save_path, 'phase5_summary.csv'), index=False)
    pd.DataFrame(freq_rows).to_csv(os.path.join(save_path, 'phase5_feature_frequency.csv'), index=False)
    pd.DataFrame({
        'fold': range(1, len(auc_50_honest) + 1),
        'auc': auc_50_honest,
        'config': CHAMPION_NAME
    }).to_csv(os.path.join(save_path, 'phase5_auc50_champion.csv'), index=False)

    return pd.DataFrame(detailed_rows), pd.DataFrame(summary_rows), auc_50_honest, pd.DataFrame(freq_rows)


In [ ]:
detailed_rows, summary_rows, auc_50, freq_df = run_phase5_final_verification()

PHASE 5: FINAL VERIFICATION — 5×10 REPEATED K-FOLD (50 folds)

  Config                                             Med       Q1       Q3  IQR  avg_feats
  --------------------------------------------------------------------------------------------
  Nested p<0.1  LR_C0.01,  all184                 0.6954   0.6034   0.7561  0.1527  31.9
  Nested p<0.1  LR_C0.001, all184                 0.7081   0.6263   0.7659  0.1396  31.9
  Nested p<0.05 LR_C0.001, all184                 0.6979   0.6314   0.7527  0.1213  21.1
  Nested p<0.1  RF_d3,     all184                 0.6654   0.6381   0.7735  0.1354  31.9
  Nested p<0.05 RF_d3,     all184                 0.6918   0.6220   0.7761  0.1541  21.1
  Nested p<0.05 LR_C0.01,  all184                 0.6978   0.6197   0.7421  0.1223  21.1
  REFERENCE: no filter, LR_C0.01, all184          0.6775   0.6057   0.7286  0.1229  184.0 (fixed)

WINNING CONFIG: Nested p<0.1  LR_C0.001, all184 (Median AUC: 0.7081)

CHAMPION ANALYTICS: Feature frequency across 10 